# Solutions 1 — Data Model, Types, and Architecture

Six worked solutions to selected internals, typing, and architecture challenges.

**Level:** Advanced · **Study mode:** Worked reference examples · **Prerequisites:** Relevant lessons and exercise workbook.

These solutions show one defensible approach each; multiple implementations may be correct. Review the contracts, test boundaries, and discuss design trade-offs before comparing line-by-line.

## 1. A hashable value object

frozen dataclasses with immutable fields make strong value keys. Equal records compare equal and produce compatible hashes.

In [ ]:
from dataclasses import dataclass

@dataclass(frozen=True)
class Money:
    cents: int
    currency: str
    def __post_init__(self):
        if not isinstance(self.cents, int) or isinstance(self.cents, bool):
            raise ValueError("cents must be an integer")
        if len(self.currency) != 3:
            raise ValueError("three-letter currency required")

a = Money(1500, "USD")
b = Money(1500, "USD")
assert a == b and hash(a) == hash(b)
assert {a: "paid"}[b] == "paid"
print(a)

**Extension:** How should arithmetic behave when currencies differ?

## 2. Independent descriptor storage

A validating descriptor stores per-instance values in each object's dictionary and never in the descriptor itself.

In [ ]:
class PositiveInt:
    def __set_name__(self, owner, name):
        self.field = name
        self.private = "_" + name
    def __get__(self, obj, owner):
        return self if obj is None else obj.__dict__[self.private]
    def __set__(self, obj, value):
        if isinstance(value, bool) or not isinstance(value, int) or value <= 0:
            raise ValueError(self.field + " must be positive int")
        obj.__dict__[self.private] = value

class Box:
    size = PositiveInt()
    def __init__(self, size):
        self.size = size

first, second = Box(1), Box(3)
first.size = 2
assert second.size == 3

**Extension:** What behavior should a descriptor have if its value has not yet been assigned?

## 3. Shallow vs deep copy

Copy outer containers separately and validate that deep copies separate nested mutable data. This does not justify deep copying real external resources.

In [ ]:
import copy

original = {"items": [{"name": "A"}]}
shallow = copy.copy(original)
deep = copy.deepcopy(original)
original["items"][0]["name"] = "B"
assert shallow["items"][0]["name"] == "B"
assert deep["items"][0]["name"] == "A"
print(original, deep)

**Extension:** How are cycles handled by deepcopy's internal memo?

## 4. A generic stack

A TypeVar binds operation input and output types in a generic container. Python type checking is static, while the runtime code still needs to handle empty-stack behavior.

In [ ]:
from typing import Generic, TypeVar
T = TypeVar("T")

class Stack(Generic[T]):
    def __init__(self):
        self._values: list[T] = []
    def push(self, item: T) -> None:
        self._values.append(item)
    def pop(self) -> T:
        if not self._values:
            raise IndexError("empty stack")
        return self._values.pop()

stack: Stack[str] = Stack()
stack.push("hello")
assert stack.pop() == "hello"

**Extension:** Why can a mutable generic container be invariant?

## 5. Safe plugin registration

Allowlisted operations avoid executing arbitrary code. Duplicate names and unsupported choices fail explicitly.

In [ ]:
class Registry:
    def __init__(self):
        self._operations = {}
    def register(self, name, operation):
        if not name or name in self._operations:
            raise ValueError("invalid or duplicate plugin")
        self._operations[name] = operation
    def run(self, name, text):
        if name not in self._operations:
            raise ValueError("unknown plugin")
        return self._operations[name](text)

registry = Registry()
registry.register("upper", str.upper)
assert registry.run("upper", "python") == "PYTHON"

**Extension:** Would a plugin imported from an untrusted source remain safe just because its name is allowlisted?

## 6. Repository injection

A service uses a small repository contract without depending on SQL or file systems. An in-memory implementation makes behavior testable.

In [ ]:
class FakeRepo:
    def __init__(self):
        self.data = {}
    def exists(self, key):
        return key in self.data
    def save(self, key, value):
        self.data[key] = value

def publish_once(repo, key, value):
    if repo.exists(key):
        raise ValueError("duplicate")
    repo.save(key, value)

repo = FakeRepo()
publish_once(repo, "A", 5)
assert repo.data == {"A": 5}

**Extension:** Why would a real concurrent repository need an atomic insert-if-absent instead?

## Recap

- Reference solutions illustrate testing and contracts rather than exhaustive answers.
- Explain one alternative design for each pattern.
- Re-run every code cell in a fresh kernel before copying into a project.

**Next:** Work independently on the unsolved questions in the corresponding workbook.